In [3]:
from typing import List
import re
from functools import partial
import requests
import pandas as pd
from typing import Final, Union, Optional
import os
from ratelimit import limits, sleep_and_retry

SEC_ARCHIVE_URL: Final[str] = "https://www.sec.gov/Archives/edgar/data"
SEC_SEARCH_URL: Final[str] = "http://www.sec.gov/cgi-bin/browse-edgar"
SEC_SUBMISSIONS_URL = "https://data.sec.gov/submissions"

def _drop_dashes(accession_number: Union[str, int]) -> str:
    """Converts the accession number to the no dash representation."""
    accession_number = str(accession_number).replace("-", "")
    return accession_number.zfill(18)

def _get_session(
    company: Optional[str] = "Indiana-University-Bloomington",
    email: Optional[str] = "athecolab@gmail.com",
) -> requests.Session:
    """Creates a requests sessions with the appropriate headers set. If these headers are not
    set, SEC will reject your request.
    ref: https://www.sec.gov/os/accessing-edgar-data"""
    if company is None:
        company = os.environ.get("SEC_API_ORGANIZATION")
    if email is None:
        email = os.environ.get("SEC_API_EMAIL")
    assert company
    assert email
    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": f"{company} {email}",
            "Content-Type": "text/html",
        }
    )
    return session

def get_filing(
    accession_number: Union[str, int], cik: Union[str, int], company: str, email: str
) -> str:
    """Fetches the specified filing from the SEC EDGAR Archives. Conforms to the rate
    limits specified on the SEC website.
    ref: https://www.sec.gov/os/accessing-edgar-data"""
    session = _get_session(company, email)
    return _get_filing(session, cik, accession_number)

def _add_dashes(accession_number: Union[str, int]) -> str:
    """Adds the dashes back into the accession number"""
    accession_number = str(accession_number)
    return f"{accession_number[:10]}-{accession_number[10:12]}-{accession_number[12:]}"

def archive_url(cik: Union[str, int], accession_number: Union[str, int]) -> str:
    """Builds the archive URL for the SEC accession number. Looks for the .txt file for the
    filing, while follows a {accession_number}.txt format."""
    filename = f"{_add_dashes(accession_number)}.txt"
    accession_number = _drop_dashes(accession_number)
    url = f"{SEC_ARCHIVE_URL}/{cik}/{accession_number}/{filename}"
    print(f"url: {url}")
    return url

@sleep_and_retry
@limits(calls=10, period=1)
def _get_filing(
    session: requests.Session, cik: Union[str, int], accession_number: Union[str, int]
) -> str:
    """Wrapped so filings can be retrieved with an existing session."""
    url = archive_url(cik, accession_number)
    # headers = {
    # 'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    # }
    company = "Indiana-University-Bloomington"
    email = "athecolab@gmail.com"
    headers = {
        "User-Agent": f"{company} {email}",
        "Content-Type": "text/html",
    }
    response = session.get(url, headers=headers)
    response.raise_for_status()
    return response.text

def _search_url(cik: Union[str, int]) -> str:
    search_string = f"CIK={cik}&Find=Search&owner=exclude&action=getcompany"
    url = f"{SEC_SEARCH_URL}?{search_string}"
    return url

@sleep_and_retry
@limits(calls=2, period=1)
def get_cik_by_ticker(ticker: str) -> str:
    """Gets a CIK number from a stock ticker by running a search on the SEC website."""
    cik_re = re.compile(r".*CIK=(\d{10}).*")
    url = _search_url(ticker)
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    # headers =  {
    # 'authority': 'www.google.com',
    # 'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    # 'accept-language': 'en-US,en;q=0.9',
    # 'cache-control': 'max-age=0',
    # 'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36',
    # # Add more headers as needed
    # }
    company = "Indiana-University-Bloomington"
    email = "athecolab@gmail.com"
    headers = {
        "User-Agent": f"{company} {email}",
        "Content-Type": "text/html",
    }
    response = requests.get(url, stream=True, headers=headers)
    response.raise_for_status()
    results = cik_re.findall(response.text)
    return str(results[0])


In [4]:
from datetime import datetime
import concurrent.futures

def sec_main(
    ticker: str,
    year: str,
    filing_types: List[str] = ["10-K", "10-Q"],
    include_amends=True,
):
    cik = get_cik_by_ticker(ticker)
    rgld_cik = int(cik.lstrip("0"))

    forms = []
    if include_amends:
        for ft in filing_types:
            forms.append(ft)
            forms.append(ft + "/A")
    print(cik)
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    # Send a GET request to the URL with headers
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        json_data = response.json()
    else:
        print(f"Error: Unable to fetch data. Status code: {response.status_code}")

    form_lists = []
    filings = json_data["filings"]
    recent_filings = filings["recent"]
    sec_form_names = []
    for acc_num, form_name, filing_date, report_date in zip(
        recent_filings["accessionNumber"],
        recent_filings["form"],
        recent_filings["filingDate"],
        recent_filings["reportDate"],
        strict=True
    ):
        if form_name in forms and report_date.startswith(str(year)):
            if form_name == "10-Q":
                datetime_obj = datetime.strptime(report_date, "%Y-%m-%d")
                quarter = pd.Timestamp(datetime_obj).quarter
                form_name += str(quarter)
                if form_name in sec_form_names:
                    form_name += "-1"
            no_dashes_acc_num = re.sub("-", "", acc_num)
            form_lists.append([no_dashes_acc_num, form_name, filing_date, report_date])
            sec_form_names.append(form_name)
    acc_nums_list = [fl[0] for fl in form_lists]
    get_filing_partial = partial(
        get_filing,
        cik=rgld_cik,
        company="Unstructured Technologies",
        email="support@unstructured.io",
    )
    print("Started Scraping")
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        results = executor.map(get_filing_partial, acc_nums_list)
    results_texts = []
    for res in results:
        results_texts.append(res)
    assert len(results_texts) == len(
        acc_nums_list
    ), f"The scraped text {len(results_texts)} is not matching with accession number texts {len(acc_nums_list)}"
    return form_lists, sec_form_names, results_texts

In [5]:
form_lists, sec_form_names, results_texts = sec_main(
    ticker="AAPL", year="2025"
)

0000320193
Started Scraping
url: https://www.sec.gov/Archives/edgar/data/320193/000032019326000006/0000320193-26-000006.txt
url: https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/0000320193-25-000079.txt
url: https://www.sec.gov/Archives/edgar/data/320193/000032019325000073/0000320193-25-000073.txt
url: https://www.sec.gov/Archives/edgar/data/320193/000032019325000057/0000320193-25-000057.txt


In [4]:
rs = results_texts[0]

In [10]:
start_marker = "PART I"
end_marker = "See accompanying Notes to Condensed Consolidated Financial Statements."
start_idx = rs.find(start_marker)
if start_idx != -1:
    extracted_text = rs[start_idx:start_idx+200_00 + len(end_marker)]
    with open("text.txt", "w", encoding="utf-8") as f:
        f.write(extracted_text)
else:
    print("Could not find the specified markers in the text.")

In [5]:
len(rs)

5191740

In [6]:
# pip install transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
checkpoint = "jinaai/reader-lm-1.5b"

device = "cuda:0" # for GPU usage or "cpu" for CPU usage
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)


/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 1134.55it/s, Materializing param=model.norm.weight]                              


In [1]:
from vllm import LLM, SamplingParams
from vllm.model_executor.models.deepseek_ocr import NGramPerReqLogitsProcessor
from PIL import Image

/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(


In [2]:
from pdf2image import convert_from_path
from pathlib import Path


def pdf_to_png(
    pdf_path: str,
    output_dir: str,
    dpi: int = 300
) -> list[Path]:
    """
    Convert a PDF into PNG images (one per page).

    Args:
        pdf_path: Path to input PDF.
        output_dir: Directory where PNGs will be saved.
        dpi: Resolution of output images.

    Returns:
        List of saved PNG file paths.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    pages = convert_from_path(pdf_path, dpi=dpi)

    saved_files: list[Path] = []

    for i, page in enumerate(pages):
        file_path = output_path / f"page_{i + 1}.png"
        page.save(file_path, "PNG")
        saved_files.append(file_path)

    return saved_files


if __name__ == "__main__":
    files = pdf_to_png("sec_data/AAPL-2025/10-Q4.pdf", "output_images")
    print(f"Converted {len(files)} pages.")

Converted 42 pages.


In [2]:
# Create model instance
llm = LLM(
    model="allenai/olmOCR-2-7B-1025",
    enable_prefix_caching=False,
    mm_processor_cache_gb=0,
    logits_processors=[NGramPerReqLogitsProcessor]
)

INFO 02-25 02:57:19 [utils.py:261] non-default args: {'enable_prefix_caching': False, 'disable_log_stats': True, 'mm_processor_cache_gb': 0, 'logits_processors': [<class 'vllm.model_executor.models.deepseek_ocr.NGramPerReqLogitsProcessor'>], 'model': 'allenai/olmOCR-2-7B-1025'}
INFO 02-25 02:57:25 [model.py:541] Resolved architecture: Qwen2_5_VLForConditionalGeneration
INFO 02-25 02:57:25 [model.py:1561] Using max model len 128000


2026-02-25 02:57:26,338	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 02-25 02:57:26 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 02-25 02:57:26 [vllm.py:624] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=1709073) INFO 02-25 02:57:31 [core.py:96] Initializing a V1 LLM engine (v0.15.0) with config: model='allenai/olmOCR-2-7B-1025', speculative_config=None, tokenizer='allenai/olmOCR-2-7B-1025', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=128000, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='',

(EngineCore_DP0 pid=1709073) The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


(EngineCore_DP0 pid=1709073) INFO 02-25 02:57:38 [gpu_model_runner.py:4021] Starting to load model allenai/olmOCR-2-7B-1025...


...

(EngineCore_DP0 pid=1709073) INFO 02-25 02:57:39 [mm_encoder_attention.py:77] Using AttentionBackendEnum.FLASH_ATTN for MMEncoderAttention.
(EngineCore_DP0 pid=1709073) INFO 02-25 02:57:39 [cuda.py:364] Using FLASH_ATTN attention backend out of potential backends: ('FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION')
(EngineCore_DP0 pid=1709073) INFO 02-25 02:58:17 [weight_utils.py:527] Time spent downloading weights for allenai/olmOCR-2-7B-1025: 37.420540 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:00<00:02,  1.25it/s]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:01<00:01,  1.03it/s]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:02<00:00,  1.29it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:03<00:00,  1.22it/s]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:03<00:00,  1.20it/s]
(EngineCore_DP0 pid=1709073) 


(EngineCore_DP0 pid=1709073) INFO 02-25 02:58:21 [default_loader.py:291] Loading weights took 3.57 seconds
(EngineCore_DP0 pid=1709073) INFO 02-25 02:58:22 [gpu_model_runner.py:4118] Model loading took 15.63 GiB memory and 42.219659 seconds
(EngineCore_DP0 pid=1709073) INFO 02-25 02:58:22 [gpu_model_runner.py:4946] Encoder cache will be initialized with a budget of 114688 tokens, and profiled with 1 video items of the maximum feature size.
(EngineCore_DP0 pid=1709073) INFO 02-25 02:58:40 [backends.py:805] Using cache directory: /home/recoverx/.cache/vllm/torch_compile_cache/fffa82a61e/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1709073) INFO 02-25 02:58:40 [backends.py:865] Dynamo bytecode transform time: 4.88 s
(EngineCore_DP0 pid=1709073) INFO 02-25 02:58:46 [backends.py:302] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=1709073) INFO 02-25 02:59:32 [backends.py:319] Compiling a graph for compile range (1, 8192) takes 49.49 s
(EngineC

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 25.62it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 31.59it/s]


(EngineCore_DP0 pid=1709073) INFO 02-25 02:59:37 [gpu_model_runner.py:5051] Graph capturing finished in 4 secs, took 0.56 GiB
(EngineCore_DP0 pid=1709073) INFO 02-25 02:59:37 [core.py:272] init engine (profile, create kv cache, warmup model) took 75.47 seconds
INFO 02-25 02:59:38 [llm.py:343] Supported tasks: ['generate']


In [ ]:
import torch
import base64
import urllib.request

from io import BytesIO
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

from olmocr.data.renderpdf import render_pdf_to_base64png
from olmocr.prompts import build_no_anchoring_v4_yaml_prompt

# Initialize the model
model = Qwen2_5_VLForConditionalGeneration.from_pretrained("allenai/olmOCR-2-7B-1025-FP8", torch_dtype=torch.bfloat16).eval()
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Grab a sample PDF
# urllib.request.urlretrieve("https://olmocr.allenai.org/papers/olmocr.pdf", "./paper.pdf")

# Render page 1 to an image
# image_base64 = render_pdf_to_base64png("./paper.pdf", 1, target_longest_image_dim=1288)

with open("sec_data/output_images/page_8.png", "rb") as img_file:
    image_bytes = img_file.read()
    image_base64 = base64.b64encode(image_bytes).decode("utf-8")

# Build the full prompt
messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": build_no_anchoring_v4_yaml_prompt()},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_base64}"}},
                ],
            }
        ]

# Apply the chat template and processor
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
main_image = Image.open(BytesIO(base64.b64decode(image_base64)))

inputs = processor(
    text=[text],
    images=[main_image],
    padding=True,
    return_tensors="pt",
)
inputs = {key: value.to(device) for (key, value) in inputs.items()}



# ['---\nprimary_language: en\nis_rotation_valid: True\nrotation_correction: 0\nis_table: False\nis_diagram: False\n---\nolmOCR: Unlocking Trillions of Tokens in PDFs with Vision Language Models\n\nJake Poz']


/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/recoverx/astarag/recursive-lm-sec-filings/.venv/lib/python3.11/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
Compressing model: 196it [00:00, 665.14it/s]
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 31.33it/s]
The image processor of type `Qwen2V

['---\nprimary_language: en\nis_rotation_valid: True\nrotation_correction: 0\nis_table: True\nis_diagram: False\n---\n<table>\n  <tr>\n    <th colspan="4">ASSETS:</th>\n  </']


In [3]:
# Generate the output
output = model.generate(
            **inputs,
            temperature=0.1,
            max_new_tokens=4096,
            num_return_sequences=1,
            do_sample=True,
        )

# Decode the output
prompt_length = inputs["input_ids"].shape[1]
new_tokens = output[:, prompt_length:]
text_output = processor.tokenizer.batch_decode(
    new_tokens, skip_special_tokens=True
)

print(text_output)

['---\nprimary_language: en\nis_rotation_valid: True\nrotation_correction: 0\nis_table: True\nis_diagram: False\n---\n<table>\n  <tr>\n    <th colspan="4">ASSETS:</th>\n  </tr>\n  <tr>\n    <th colspan="2">Current assets:</th>\n    <th>December 27, 2025</th>\n    <th>September 27, 2025</th>\n  </tr>\n  <tr>\n    <td>Cash and cash equivalents</td>\n    <td>$ 45,317</td>\n    <td>$ 35,934</td>\n  </tr>\n  <tr>\n    <td>Marketable securities</td>\n    <td>21,590</td>\n    <td>18,763</td>\n  </tr>\n  <tr>\n    <td>Accounts receivable, net</td>\n    <td>39,921</td>\n    <td>39,777</td>\n  </tr>\n  <tr>\n    <td>Vendor non-trade receivables</td>\n    <td>30,399</td>\n    <td>33,180</td>\n  </tr>\n  <tr>\n    <td>Inventories</td>\n    <td>5,875</td>\n    <td>5,718</td>\n  </tr>\n  <tr>\n    <td>Other current assets</td>\n    <td>15,002</td>\n    <td>14,585</td>\n  </tr>\n  <tr>\n    <td>Total current assets</td>\n    <td>158,104</td>\n    <td>147,957</td>\n  </tr>\n  <tr>\n    <th colspan="2"

In [ ]:
print(text_output[0])

---
primary_language: en
is_rotation_valid: True
rotation_correction: 0
is_table: True
is_diagram: False
---
<table>
  <tr>
    <th colspan="4">ASSETS:</th>
  </tr>
  <tr>
    <th colspan="2">Current assets:</th>
    <th>December 27, 2025</th>
    <th>September 27, 2025</th>
  </tr>
  <tr>
    <td>Cash and cash equivalents</td>
    <td>$ 45,317</td>
    <td>$ 35,934</td>
  </tr>
  <tr>
    <td>Marketable securities</td>
    <td>21,590</td>
    <td>18,763</td>
  </tr>
  <tr>
    <td>Accounts receivable, net</td>
    <td>39,921</td>
    <td>39,777</td>
  </tr>
  <tr>
    <td>Vendor non-trade receivables</td>
    <td>30,399</td>
    <td>33,180</td>
  </tr>
  <tr>
    <td>Inventories</td>
    <td>5,875</td>
    <td>5,718</td>
  </tr>
  <tr>
    <td>Other current assets</td>
    <td>15,002</td>
    <td>14,585</td>
  </tr>
  <tr>
    <td>Total current assets</td>
    <td>158,104</td>
    <td>147,957</td>
  </tr>
  <tr>
    <th colspan="2">Non-current assets:</th>
    <th></th>
    <th></th>


: 